# DE0831 Project 2, track E (log mining) on Google Colab — Vagif Asgarli, 251ADM047

1. **Runtime → Change runtime type → Python 3, T4 GPU**, then **Runtime → Run all**.
2. When the upload button appears (step 3), choose `DE0831_P2.zip`.
3. Installing CUDA.jl and CairoMakie takes 10–20 minutes the first time; generating the six
   corpora and running the benchmarks takes about 20 minutes. At the end the browser downloads
   `DE0831_P2_Asgarli_251ADM047.zip` (for ORTUS) and `run_logs.zip`.

Every step stops with an error if anything goes wrong; the run script prints `Correct!` when
all validation checks pass.

If a run fails and a corrected `DE0831_P2.zip` is provided: run step 3 again (the upload replaces
the source files, the generated data stays), then **Runtime → Run after** on the step 5 cell.


## 1. GPU

In [ ]:
!nvidia-smi

## 2. Install Julia 1.12.7 (the version the Manifest was resolved with)

In [ ]:
%%bash
set -e
cd /content
if [ ! -x /content/julia-1.12.7/bin/julia ]; then
  wget -q https://julialang-s3.julialang.org/bin/linux/x64/1.12/julia-1.12.7-linux-x86_64.tar.gz
  tar xzf julia-1.12.7-linux-x86_64.tar.gz
fi
/content/julia-1.12.7/bin/julia --version

## 3. Upload the project (`DE0831_P2.zip`)

In [ ]:
import os, zipfile
from google.colab import files
print('Choose DE0831_P2.zip. A newer zip can be uploaded over an existing /content/DE0831_P2:',
      'its source files are replaced and the generated data/ is kept.')
uploaded = files.upload()
for name in uploaded:
    zipfile.ZipFile(name).extractall('/content')
    print('extracted', name)
assert os.path.isdir('/content/DE0831_P2'), 'DE0831_P2.zip was not uploaded'
print(sorted(os.listdir('/content/DE0831_P2')))


## 4. Install packages (CUDA.jl, CairoMakie) and check CUDA

In [ ]:
import os, subprocess

def run(cmd, log):
    """Run a shell command, show its output live in the cell and save it to a log file."""
    os.makedirs('/content/logs', exist_ok=True)
    with open(log, 'w') as f:
        p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
        p.wait()
    assert p.returncode == 0, f'command failed (exit {p.returncode}), see the output above or {log}'


In [ ]:
run("cd /content/DE0831_P2 && /content/julia-1.12.7/bin/julia --project=. -e 'using Pkg; Pkg.instantiate(); using CUDA; CUDA.versioninfo(); @assert CUDA.functional()'", '/content/logs/instantiate.log')

## 5. Full run: validation, benchmarks over six corpus sizes, results, plots

In [ ]:
run('cd /content/DE0831_P2 && /content/julia-1.12.7/bin/julia --project=. src/p2_run.jl', '/content/logs/p2_run.log')

## 6. Test suite (validation only; must print `Correct!`)

In [ ]:
run('cd /content/DE0831_P2 && /content/julia-1.12.7/bin/julia --project=. test/runtests.jl', '/content/logs/runtests.log')

## 7. Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('/content/DE0831_P2/README.md').read().split('## Individualisation')[0]))
display(Markdown(open('/content/DE0831_P2/results/benchmarks.md').read()))
for png in ['throughput_vs_size', 'time_breakdown', 'entropy_distribution', 'threshold_precision_recall']:
    p = f'/content/DE0831_P2/results/{png}.png'
    if os.path.exists(p): display(Image(p))
display(Markdown(open('/content/DE0831_P2/results/analysis.md').read()))

## 8. Build and download the submission archive

In [ ]:
import shutil
name = 'DE0831_P2_Asgarli_251ADM047'
dst = '/content/' + name
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree('/content/DE0831_P2', dst, ignore=shutil.ignore_patterns('data', '.gitignore', '*.zip'))
archive = shutil.make_archive(dst, 'zip', '/content', name)
shutil.make_archive('/content/run_logs', 'zip', '/content/logs')
print(archive)
files.download(archive)
files.download('/content/run_logs.zip')